<!-- AI-og-helse: colab-kontrakt v1 -->

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/AI-og-helse/blob/main/utils/imgur-opplasting.ipynb)

## Colab-kjøring

- **Anbefalt runtime:** CPU
- **Forventet kjøretid:** 5-10 min
- **Datamønster:** `local-utility`

**Krav før kjøring:**
- API/data: utility-notebook, kan kreve lokal filtilpasning
- GPU: ikke nødvendig

**Felles konvensjon:** Kjør setup-cellen rett under først. I Colab hentes hemmeligheter fra **Secrets** med `userdata.get(...)`; lokalt brukes miljøvariabler eller `.env`.

Dette er en hjelpe-notebook og er ikke en primær kursnotebook.


In [ ]:
# AI-og-helse: Colab bootstrap v1
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
NOTEBOOK_PACKAGES = []
COLAB_DATA_MODE = 'local-utility'


def _has_import(import_name: str) -> bool:
    return importlib.util.find_spec(import_name) is not None


def ensure_packages(packages=NOTEBOOK_PACKAGES):
    """Install only notebook-specific packages when running in Colab."""
    if not IN_COLAB:
        return

    missing = [package for package, import_name in packages if not _has_import(import_name)]
    if missing:
        print("Installerer Colab-pakker:", ", ".join(missing))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("Alle notebook-spesifikke Colab-pakker er tilgjengelige.")


def get_secret(name: str, *aliases: str):
    """Read secrets from environment/.env locally or Colab Secrets in Colab."""
    for key in (name, *aliases):
        value = os.getenv(key)
        if value:
            os.environ[name] = value
            return value

    if IN_COLAB:
        try:
            from google.colab import userdata
        except Exception:
            userdata = None

        if userdata is not None:
            for key in (name, *aliases):
                try:
                    value = userdata.get(key)
                except Exception:
                    value = None
                if value:
                    os.environ[name] = value
                    return value

    return None


def mount_drive_if_needed():
    """Mount Google Drive explicitly in notebooks that need persistent artifacts."""
    if not IN_COLAB:
        return None
    from google.colab import drive

    drive.mount("/content/drive")
    return Path("/content/drive/MyDrive")


ensure_packages()
print("Miljø:", "Google Colab" if IN_COLAB else "lokalt")
print("Datamønster:", COLAB_DATA_MODE)


📤 Automatisk Imgur-opplasting

In [3]:
# imgur-opplasting.ipynb

import requests
import base64
import json
from IPython.display import Image, display
import os

def last_opp_til_imgur(fil_sti, client_id=None):
    """
    Last opp bilde til Imgur og returner permanent lenke
    
    Args:
        fil_sti: Sti til bildefilen
        client_id: Imgur API Client ID (valgfri for anonym opplasting)
    
    Returns:
        dict: Imgur respons med lenker
    """
    
    # Sjekk om filen eksisterer
    if not os.path.exists(fil_sti):
        print(f"❌ Finner ikke fil: {fil_sti}")
        return None
    
    # Les og konverter bildet til base64
    with open(fil_sti, 'rb') as f:
        image_data = f.read()
    
    # Konverter til base64
    encoded_image = base64.b64encode(image_data).decode()
    
    # Imgur API endpoint
    url = "https://api.imgur.com/3/image"
    
    # Headers (anonym opplasting hvis ingen client_id)
    headers = {
        'Authorization': f'Client-ID {client_id}' if client_id else 'Client-ID 546c25a59c58ad7'
    }
    
    # Data for opplasting
    data = {
        'image': encoded_image,
        'type': 'base64',
        'title': 'Dyplærings-modell Topol 2019',
        'description': 'AI og Helse kurs - Topol 2019 dyplæringsmodell'
    }
    
    print("📤 Laster opp til Imgur...")
    
    try:
        # Send POST request
        response = requests.post(url, headers=headers, data=data)
        
        if response.status_code == 200:
            result = response.json()
            
            if result['success']:
                data = result['data']
                
                print("✅ Opplasting vellykket!")
                print(f"🔗 Imgur URL: {data['link']}")
                print(f"📋 Delete hash (for sletting): {data['deletehash']}")
                
                # Vis bildet
                display(Image(url=data['link'], width=800))
                
                return {
                    'url': data['link'],
                    'delete_url': f"https://imgur.com/delete/{data['deletehash']}",
                    'deletehash': data['deletehash']
                }
            else:
                print(f"❌ Imgur feil: {result}")
                return None
        else:
            print(f"❌ HTTP feil: {response.status_code}")
            print(f"Response: {response.text}")
            return None
            
    except Exception as e:
        print(f"❌ Opplastingsfeil: {e}")
        return None


In [4]:
# Last opp bildet
from pathlib import Path

# Utility-notebook: bruk repo-relativ sti når notebooken kjøres fra `utils/`.
fil_sti = Path.cwd().parent / "uke01-introduksjon" / "ressurser" / "dyplærings-modell-topol-2019.png"
imgur_result = last_opp_til_imgur(str(fil_sti))

if imgur_result:
    print("\n" + "="*60)
    print("🎯 BRUK DENNE LENKEN I DINE NOTEBOOKS:")
    print("="*60)
    print(f"imgur_url = '{imgur_result['url']}'")
    print("display(Image(url=imgur_url, width=800))")
    print("="*60)

📤 Laster opp til Imgur...
✅ Opplasting vellykket!
🔗 Imgur URL: https://i.imgur.com/2AV2Kfb.png
📋 Delete hash (for sletting): 3yjzyAMGfDBFu4f



🎯 BRUK DENNE LENKEN I DINE NOTEBOOKS:
imgur_url = 'https://i.imgur.com/2AV2Kfb.png'
display(Image(url=imgur_url, width=800))
